In [1]:
%matplotlib widget
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from ipywidgets import IntSlider, interact

rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_csv("../data/raw/may_filtered.csv")
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S').astype("datetime64[s]")
df.drop(['date', 'time'], inplace=True, axis=1)
df['motion'] = np.linalg.norm(df[['accX', 'accY', 'accZ']].values, axis=1)

df['duration_td'] = pd.to_timedelta(df['duration'])
df['duration_sec'] = df['duration_td'].dt.total_seconds()

In [ ]:
y = -1. * df['ied'].values
ymin, ymax = np.nanmin(y), np.nanmax(y)
plt.close('all')
@interact(
    start=IntSlider(min=0, max=len(y)-200, step=30, value=0, description='start idx'),
    window=IntSlider(min=200, max=7900000, step=200, value=1000, description='window')
)
def plot_zoom(start=0, window=500):
    end = min(start + window, len(y))
    t = np.arange(start, end) / 100.0
    plt.rcParams['font.size'] = 18
    fig, ax = plt.subplots(figsize=(18, 7))
    ax.plot(t, y[start:end], 'o-', linewidth=1, markersize=2)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('IED (high-pass)')
    # ax.set_ylim([ymin, ymax])

    ax.set_title(f'Sample {start}~{end} ({window} samples / {window/100:.1f}s)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.signal import medfilt

In [ ]:
wsize = 15
f_sig = medfilt(y, kernel_size=wsize)

In [ ]:
time = df.datetime
time

In [ ]:
len(f_sig)

In [ ]:
Y = pd.DataFrame(
    {
        'time':time,
        'smoothed_ied':f_sig
    }
)

In [ ]:
Y.dtypes

In [ ]:
# 时间轴 存在 极短断层（间隔 2~5 秒） 中等断层（几十秒到几分钟）  超长断层（几十分钟到数小时

In [ ]:
def segment_continuous_data(df, time_col='time', max_gap=5, min_samples=50):
    t = df[time_col].astype('int64')
    d = np.diff(t)
    break_indices = np.where(d > max_gap)[0]
    start_idx = 0
    for break_idx in break_indices:
        end_idx = break_idx + 1
        if end_idx - start_idx >= min_samples:
            yield df.iloc[start_idx:end_idx].copy()
        start_idx = end_idx
    if len(df) - start_idx >= min_samples:
        yield df.iloc[start_idx:].copy()

In [ ]:
segment_generator = segment_continuous_data(Y)
seg = segment_generator.__next__()
seg

In [ ]:
import neurokit2 as nk

In [ ]:
cleaned_sig = nk.ppg_clean(f_sig, sampling_rate=100)
info = nk.ppg_findpeaks(cleaned_sig, sampling_rate=100)

In [ ]:
size_freq = df.groupby("datetime").size().value_counts().sort_index()

In [ ]:
# print(size_freq.to_string())

In [ ]:
# 假设你的 DataFrame 变量名为 df
# 确保 datetime 列按时间排序
df = df.sort_values('datetime').reset_index(drop=True)

# ---------------------------------------------------------
# 步骤 1：检测物理断点，划分子段 (Chunking)
# ---------------------------------------------------------
# 计算相邻行之间的秒级时间差
time_diff = df['datetime'].diff().dt.total_seconds()

# 设定断点阈值：考虑到网络拥塞可能导致某几秒没收到数据，
# 我们将阈值放宽。如果相邻两行的数据秒数跳跃超过 5 秒，我们认为这是一个“物理断连”
gap_threshold = 1.5

# 生成子段 ID：每次时间差大于阈值，ID 累加 1
df['chunk_id'] = (time_diff > gap_threshold).cumsum()

print(f"数据被划分为 {df['chunk_id'].nunique()} 个连续子段。")

# ---------------------------------------------------------
# 步骤 2：在每个子段内部，基于起始时间和行号，生成绝对均匀的 100Hz 时间轴
# ---------------------------------------------------------
def generate_ideal_time(chunk):
    # 取该段的第一行 datetime 作为物理锚点
    start_time = chunk['datetime'].iloc[0]
    
    # 严格按照当前子段的行数，以 10ms (100Hz) 为步长生成时间戳
    ideal_index = pd.date_range(start=start_time, periods=len(chunk), freq='10ms')
    
    # 将生成的高精度时间戳赋给该段
    chunk['ideal_time'] = ideal_index
    return chunk

# 应用函数
df = df.groupby('chunk_id', group_keys=False).apply(generate_ideal_time)

# ---------------------------------------------------------
# 步骤 3：验证时钟漂移 (Clock Drift Sanity Check)
# ---------------------------------------------------------
# 理论上，生成的 ideal_time 的秒级部分，应该和原始的 datetime 差不多。
# 如果一段数据长达十几个小时，硬件时钟可能会有一点点温漂。
df['drift_seconds'] = (df['ideal_time'].dt.floor('s') - df['datetime']).dt.total_seconds()

print("\n--- 时钟漂移统计 (秒) ---")
print(df['drift_seconds'].describe())

# 最后，将完美的时间轴设为索引
df = df.set_index('ideal_time')

# 丢弃不再需要的辅助列
df = df.drop(columns=['duration', 'datetime', 'chunk_id', 'drift_seconds'], errors='ignore')